In [1]:
import logging
import re
import textwrap
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import gurobipy as gp
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib_venn as mpl_venn
import numpy as np
import optlang
import pandas as pd
import seaborn as sns
import sympy
from cobra import DictList, Reaction
from cobra.flux_analysis.variability import (find_blocked_reactions,
                                             flux_variability_analysis)
from cobra.util.array import create_stoichiometric_matrix, nullspace
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rbc_gem_utils import (ANNOTATION_PATH, COBRA_CONFIGURATION, CURATION_PATH,
                           DATABASE_PATH, GEM_NAME, INTERIM_PATH,
                           PARAMETERIZATION_PATH, PROCESSED_PATH, ROOT_PATH,
                           build_string, check_database_version_online,
                           check_version, compare_tables, explode_column,
                           get_annotation_df, read_cobra_model, read_rbc_model,
                           show_versions, split_string, visualize_comparison,
                           write_cobra_model)
from rbc_gem_utils.analysis.overlay import *
from rbc_gem_utils.database.uniprot import (UNIPROT_DB_TAG,
                                            UNIPROT_ISOFORM_ID_RE,
                                            UNIPROT_PATH)
from rbc_gem_utils.qc import reset_reaction_bounds, reset_subsystem_groups
from rbc_gem_utils.util import (AVOGADRO_NUMBER, DEFAULT_DRY_MASS_PER_CELL,
                                convert_gDW_to_L, convert_L_to_gDW,
                                ensure_iterable, log_msg, strip_plural)
from rbc_gem_utils.visualization import cmap_map
from scipy.stats import spearmanr
from sklearn.metrics import r2_score
from sympy import parse_expr

gp.setParam("OutputFlag", 0)
gp.setParam("LogToConsole", 0)

# Show versions of notebook
show_versions()
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Arial"

Set parameter Username

Package Information
-------------------
rbc-gem-utils 0.0.1

Dependency Information
----------------------
beautifulsoup4                       4.12.3
bio                                 missing
cobra                                0.29.1
depinfo                               2.2.0
gurobipy                             11.0.3
matplotlib                           3.10.0
matplotlib-venn                       1.1.1
memote                               0.17.0
networkx                              3.4.2
notebook                              7.3.2
openpyxl                              3.1.5
pandas                                2.2.3
pre-commit                            4.1.0
rbc-gem-utils[database,network,vis] missing
requests                             2.32.3
scipy                                1.15.1
seaborn                              0.13.2

Build Tools Information
-----------------------
pip          24.2
setuptools 75.1.0
wheel      0.44.0

Platform Informat

In [2]:
COBRA_CONFIGURATION.solver = "gurobi"
# Set bound defaults much larger to prevent model loading issues
COBRA_CONFIGURATION.bounds = (-1e8, 1e8)
COBRA_CONFIGURATION

Attribute,Description,Value
solver,Mathematical optimization solver,gurobi
tolerance,"General solver tolerance (feasibility, integrality, etc.)",1e-07
lower_bound,Default reaction lower bound,-100000000.0
upper_bound,Default reaction upper bound,100000000.0
processes,Number of parallel processes,127
cache_directory,Path for the model cache,C:\Users\Alicia Key\AppData\Local\opencobra\cobrapy\Cache
max_cache_size,Maximum cache size in bytes,104857600
cache_expiration,Model cache expiration time in seconds (if any),None


In [3]:
model_id = "RBC3P_expanded"
dataset_name = "RBComics_G6PD"
ftype = "xml"

root_path = Path("../../../..").resolve()
results_path = root_path / "data" / "processed" / model_id / "OVERLAY"
dataset_path = results_path / dataset_name
pcfva_results_dirpath = dataset_path / "pcFVA"
dataset_models_dirpath = dataset_path / "pcmodels"

print(root_path)
print(results_path)
print(dataset_path)
print(pcfva_results_dirpath)
print(dataset_models_dirpath)

D:\Projects\RBC-GEM-akey7
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\pcFVA
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\pcmodels


## Broken models are:

1. RBC3P_expanded_PC_S00163_D10
2. RBC3P_expanded_PC_S00400_D42

### RBC3P_expanded_PC_S00163_D10

In [4]:
pcmodel_sample = "RBC3P_expanded_PC_S00163_D10"
pcmodel_sample_filename = dataset_models_dirpath / f"{pcmodel_sample}.{ftype}"
print(pcmodel_sample_filename)
print(pcmodel_sample_filename.exists())

D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\pcmodels\RBC3P_expanded_PC_S00163_D10.xml
True


In [5]:
pcmodel_sample = load_overlay_model(filename=pcmodel_sample_filename)
pcmodel_sample

Name,RBC3P_expanded_PC_S00163_D10
Memory address,1f0cd0c34a0
Number of metabolites,516
Number of reactions,1080
Number of genes,107
Number of groups,12
Objective expression,1.0*NaKt - 1.0*NaKt_reverse_db47e
Compartments,"cytosol, extracellular space, protein compartment"


When optimization with the chosen slack value is attempted, it fails.

In [6]:
print(pcmodel_sample.objective)

Maximize
1.0*NaKt - 1.0*NaKt_reverse_db47e


In [8]:
chosen_slack_var = 0.006
update_slack_value(pcmodel_sample, slack_value=chosen_slack_var, verbose=True)
pcmodel_sample.optimize()

Relaxation budget updated for RBC3P_expanded_PC_S00163_D10, extra 5.7526 mg/gDW (5.7000 mg HB/gDW) from 0.6000% slack


d:\Anaconda3\envs\RBC-GEM-akey7\Lib\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


<Solution infeasible at 0x1f0cdbf3e30>

Update the slack value and attempt solutions.

In [9]:
chosen_slack_var = 0.01
update_slack_value(pcmodel_sample, slack_value=chosen_slack_var, verbose=True)

Relaxation budget updated for RBC3P_expanded_PC_S00163_D10, extra 9.5876 mg/gDW (9.5000 mg HB/gDW) from 1.0000% slack
